In [1]:
import sys
print(sys.executable)

c:\Users\alang\Documents\GitHub\COurseIronhack\IronHack_Course_Repo\Prompt Engineering Lab\lab-prompt-engineering\.venv\Scripts\python.exe


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env", override=True)

print("API key gevonden:", bool(os.getenv("OPENAI_API_KEY")))

API key gevonden: True


In [3]:
sentiment_prompt_v1 = """
Evaluate whether the customer is very satisfied.
Timely delivery and a product that works as expected are positive.
Delays, faulty products or unmet expectations are negative.
comments not referring ot the product or service should be classified as neutral
Customer message:
"The delivery was fast, but the product stopped working after one day."
"""

In [5]:
from openai import OpenAI

client = OpenAI()
print("Client klaar")

Client klaar


In [6]:
response = client.responses.create(
    model="gpt-4o-mini",
    input=sentiment_prompt_v1,
)

print(response.output_text)

Based on the customer message, the evaluation of satisfaction is mixed. 

- The **timely delivery** is a positive aspect.
- However, the **product stopped working after one day**, which is a significant negative aspect.

Overall, the negative experience with the product outweighs the positive delivery experience, suggesting that the customer is **not very satisfied**.


### Eerste observatie — sentiment v1
Het model herkent de positieve levering en het defecte product.
Het geeft een gemengd oordeel met uitleg, in plaats van één classificatielabel.

In [44]:
def call_openai(prompt):
    response = client.responses.create(
        model="gpt-4o-mini",
        input=prompt,
    )
    return response.output_text

In [45]:
sentiment_results_v1_5 = []

for i in range(5):
    result = call_openai(sentiment_prompt_v1)
    sentiment_results_v1_5.append(result)
    print(f"\n--- Test {i + 1} ---")
    print(result)


--- Test 1 ---
Based on the customer message, the evaluation is as follows:

- **Timely delivery**: Positive
- **Product stopped working after one day**: Negative

Overall, the customer seems dissatisfied due to the product issue, despite the positive aspect of fast delivery. Therefore, the customer is **not very satisfied**.

--- Test 2 ---
To evaluate the customer's satisfaction:

1. **Positive Aspects**: The customer mentions that "the delivery was fast," which is a positive comment about timeliness.
2. **Negative Aspects**: The significant negative aspect is that "the product stopped working after one day," indicating a faulty product and unmet expectations.

Overall, the negative aspect regarding the product outweighs the positive comment about delivery. Therefore, the customer is **not very satisfied**.

--- Test 3 ---
The customer expressed a positive aspect by noting that "the delivery was fast," which reflects timely delivery. However, they also mentioned a significant negati

antwoorden te lang, labels te uitgebreid en te onregelmatig


In [46]:
sentiment_prompt_v2 = """
Evaluate whether the customer is satisfied.
Timely delivery and a product that works as expected are positive.
Delays, faulty products or unmet expectations are negative.
comments not referring ot the product or service should be classified as neutral
"Return a label: positive, neutral, or negative. one classification, one label and one short description very brief and to the point in your answer. 
"The delivery was fast, but the product stopped working after one day."
Make sure that you only display in this format:
label: <positive/neutral/negative>
description: <your short description here>
In the description, briefly state the main issue or positive experience explicitly mentioned in the message.Base the description only on the customer message. Do not infer whether the product is usable unless explicitly stated.
also, dont repeat that the customer is dissatisfied, that is already represented by the label. Do not add any other information or text to your answer."""


In [86]:
sentiment_prompt_v3 = """
Classify customer feedback as positive, neutral or negative.
For mixed feedback, classify as negative if a product fault is reported.
Use only facts stated in the message.
Return exactly two lines: label and description.
Keep the description to one sentence of at most 15 words.

Examples:

Message: "The headphones work perfectly. I love them."
label: positive
description: The headphones work perfectly.

Message: "My order number is 456."
label: neutral
description: The customer provides an order number without expressing an opinion.

Message: "Delivery was quick, but the screen is broken."
label: negative
description: The screen is broken despite quick delivery.

Now classify this message:
"The delivery was fast, but the product stopped working after one day."
"""

print(call_openai(sentiment_prompt_v3))

label: negative  
description: The product stopped working after one day despite fast delivery.


In [47]:
sentiment_results_v2 = []

for i in range(5):
    result = call_openai(sentiment_prompt_v2)
    sentiment_results_v2.append(result)
    print(f"\n--- Test {i + 1} ---")
    print(result)


--- Test 1 ---
label: negative  
description: The product stopped working after one day.

--- Test 2 ---
label: negative  
description: The product stopped working after one day despite fast delivery.

--- Test 3 ---
label: negative  
description: The product stopped working after one day.

--- Test 4 ---
label: negative  
description: The product stopped working after one day.

--- Test 5 ---
label: negative  
description: The product stopped working after one day despite fast delivery.


In [88]:
sentiment_results_v3_15 = []

for i in range(15):
    result = call_openai(sentiment_prompt_v3)
    sentiment_results_v3_15.append(result)
    print(f"\n--- Test {i + 1} ---")
    print(result)


--- Test 1 ---
label: negative  
description: The product stopped working after one day despite fast delivery.

--- Test 2 ---
label: negative  
description: The product stopped working after one day despite fast delivery.

--- Test 3 ---
label: negative  
description: The product stopped working after one day despite fast delivery.

--- Test 4 ---
label: negative  
description: The product stopped working after one day despite fast delivery.

--- Test 5 ---
label: negative  
description: The product stopped working after one day despite fast delivery.

--- Test 6 ---
label: negative  
description: The product stopped working after one day despite fast delivery.

--- Test 7 ---
label: negative  
description: The product stopped working after one day despite fast delivery.

--- Test 8 ---
label: negative  
description: The product stopped working after one day despite fast delivery.

--- Test 9 ---
label: negative  
description: The product stopped working after one day despite fast de

15/15 geslaagd voor label, formaat en lengte: 100%.  

In [89]:
for i, result in enumerate(sentiment_results_v3_15, start=1):
    lines = [line.strip() for line in result.splitlines() if line.strip()]

    correct_format = (
        len(lines) == 2
        and lines[0] == "label: negative"
        and lines[1].startswith("description:")
    )

    word_count = (
        len(lines[1].removeprefix("description:").split())
        if correct_format else 0
    )

    passed = correct_format and 1 <= word_count <= 15
    print(f"Test {i}: {'PASS' if passed else 'FAIL'} | {word_count} woorden")

Test 1: PASS | 10 woorden
Test 2: PASS | 10 woorden
Test 3: PASS | 10 woorden
Test 4: PASS | 10 woorden
Test 5: PASS | 10 woorden
Test 6: PASS | 10 woorden
Test 7: PASS | 10 woorden
Test 8: PASS | 10 woorden
Test 9: PASS | 10 woorden
Test 10: PASS | 10 woorden
Test 11: PASS | 10 woorden
Test 12: PASS | 10 woorden
Test 13: PASS | 10 woorden
Test 14: PASS | 10 woorden
Test 15: PASS | 10 woorden


In [ ]:
##OK lest goooo into production fase nigggagaahah

SyntaxError: invalid syntax (174770362.py, line 1)

In [53]:
product_prompt_v1 = """
 BRIEF AND LOW VERBOSITY, mention features, competitor advantages, price advantages or payment options if explicitly provided in the product data.
 een korte productbeschrijving met weinig reclametaal.
 alleen bevestigde informatie, niks zelf verzinnen.
 ALLEEN in het Nederlands.

 DISPLAY HET ALTIJD IN DEZE FORMAT:
 PRODUCTNAAM 
 PRIJS in de oorspronkelijke valuta, niets omrekenen naar euro's of dollars, geen valuta symbolen, alleen de prijs.
 LEVERTIJD niet verzinen, alleen als het expliciet in de productdata staat.

Product: Wireless mouse en Price: $29.99
"""

print(call_openai(product_prompt_v1))

WIRELESS MOUSE  
PRIJS 29.99  
LEVERTIJD geen informatie


De productprompt levert onbevestigde eigenschappen en levertijd op en verandert de valuta, ondanks de instructie om niets te verzinnen.

In [60]:
product_prompt_v3 = """
 BRIEF AND LOW VERBOSITY, mention features, competitor advantages, price advantages or payment options if explicitly provided in the product data.
 een korte productbeschrijving met weinig reclametaal.
 alleen bevestigde informatie, niks zelf verzinnen.
 ALLEEN in het Nederlands.

 DISPLAY HET ALTIJD IN DEZE FORMAT:
 PRODUCTNAAM 
PRIJS: behoud de oorspronkelijke valuta als valutacode, bijvoorbeeld USD 29.99.
LEVERTIJD: schrijf "Niet opgegeven" als deze ontbreekt.
BESCHRIJVING: schrijf één korte Nederlandse zin op basis van de beschikbare productgegevens. Voeg geen onbevestigde eigenschappen toe.n".
Elke eigenschap in de beschrijving moet expliciet in de productdata staan. Voorkom herhalen van prijs of andere feiten die al in de andere velden staan. Voeg geen extra informatie toe die niet in de productgegevens staat.
“Wireless” bevestigt alleen dat de muis draadloos is, niet dat hij ergonomisch is.


Product: Wireless mouse en Price: $29.99
"""

print(call_openai(product_prompt_v3))

**WIRELESS MOUSE**  
PRIJS: USD 29.99  
LEVERTIJD: Niet opgegeven  
BESCHRIJVING: Draadloze muis.


In [ ]:
##Reductiefeedback

In [ ]:
extraction_prompt_v1 = """
Extract the following information from the product data:
in this format line by line, no extra text, no explanations, no apologies, no additional information, no extra formatting, no markdown, no code blocks, no quotes, just the raw text in the specified format:

clientid=
itemnumber=
date        =dd/mm/yyyy
delivery_feedback=
product_feedback=
packaging_feedback=
Ai model used= you have the mention which model was used for this query, for example gpt-4o-mini or gpt-4o
amount of tokens used= you have to make a approximate calcuation of the amount of tokens used for this query, based on the length of the prompt and the length of the response, and write it in this format: "approximate number of tokens used for this query = <number>"
price of the model usage for this query = define the calculation of the price of the model usage for this query in USD and EUR, based on the amount of tokens used and the pricing of the model used. Write it in this format: "price of the model usage for this query in USD= <amount>" and "price of the model usage for this query in EUR accumlated sum fornthis month= <amount>"
price of the model usage for this query in USD= define the calculation of the price of the model usage for this query in USD, based on the amount of tokens used and the pricing of the model used. Write it in this format: "price of the model usage for this query in USD= <amount>"
price of the model usage for this query in EUR accumlated sum fornthis month= define the calculation of the price of the model usage for this query in EUR, based on the amount of tokens used and the pricing of the model used. Write it in this format: "price of the model usage for this query in EUR accumlated sum fornthis month= <amount>"
assign this feedback to a category: define the category of the feedback based on the customer message, for example "delivery", "product", "packaging", "service", "other". If the feedback is about multiple categories, choose the most relevant one. If it is not clear, write "Not provided".
assignthis feedback to a customer agnet: define the customer agent based on the customer message, for example "delivery", "product", "packaging", "service", "other". If the feedback is about multiple agents, choose the most relevant one. If it is not clear, write "Not provided".
propose further action for our customer service team: define the further action for our customer service team based on the customer message, for example "contact the customer", "send a replacement", "issue a refund", "investigate the issue", "not required". If it is not clear, write "Not provided".
dont invent any information, only extract what is explicitly stated in the customer message. If a piece of information is not present, write "Not provided" for that field.
time required to generate this response= define the approximate time required to generate the response in seconds, based on the length of the prompt and the length of the response, and write it in this format: "approximate time required to generate the response = <number> seconds"


“I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged.”
"""

print(call_openai(extraction_prompt_v1))

clientid=Not provided  
itemnumber=12345  
date        =15/03/2023  
delivery_feedback=fast  
product_feedback=Not provided  
packaging_feedback=damaged  
Ai model used=gpt-4o  
approximate number of tokens used for this query = 96  
price of the model usage for this query in USD=0.00144  
price of the model usage for this query in EUR accumulated sum for this month=0.00135  
assign this feedback to a category=packaging  
assign this feedback to a customer agent=packaging  
prose further action for our customer service team=investigate the issue  
approximate time required to generate the response = 5 seconds  


In [67]:
extraction_prompt_v2 = """
Extract only information explicitly stated in the customer message.

Return exactly these six fields, one per line:
clientid=<value>
itemnumber=<value>
date=<value>
delivery_feedback=<value>
product_feedback=<value>
packaging_feedback=<value>

Rules:
- If information is missing, write "Not provided".
- Do not infer product damage from damaged packaging.
- If a complete date is given, use dd/mm/yyyy.
- If the year is missing, preserve the date as stated. Never invent a year.
- Do not add explanations, Markdown, or additional fields.

Customer message:
"I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

print(call_openai(extraction_prompt_v2))

clientid=Not provided  
itemnumber=12345  
date=15/03  
delivery_feedback=fast  
product_feedback=Not provided  
packaging_feedback=damaged  


In [69]:
extraction_prompt_v3 = """
Extract only information explicitly stated in the customer message.

Return exactly these six fields, one per line:
clientid=<value>
itemnumber=<value>
date=<value>
delivery_feedback=<value>
product_feedback=<value>
packaging_feedback=<value>
category=<value>
suggested_team=<value>
proposed_action=<value>
price of the model usage for this query in USD= <amount>
price of the model usage for this query in EUR accumlated sum fornthis month= <amount>

Rules:
- If information is missing, write "Not provided".
- Do not infer product damage from damaged packaging.
- If a complete date is given, use dd/mm/yyyy.
- If the year is missing, preserve the date as stated. Never invent a year.
- Do not add explanations, Markdown, or additional fields.

Customer message:
"I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

print(call_openai(extraction_prompt_v3))

clientid=Not provided  
itemnumber=12345  
date=15/03  
delivery_feedback=fast  
product_feedback=Not provided  
packaging_feedback=damaged  
category=Not provided  
suggested_team=Not provided  
proposed_action=Not provided  
price of the model usage for this query in USD= Not provided  
price of the model usage for this query in EUR accumlated sum fornthis month= Not provided  


In [72]:
extraction_prompt_v4 = """
Extract only information explicitly stated in the customer message.

Return exactly nine fields, one per line: six extracted fields followed by three assessment fields.clientid=<value>
itemnumber=<value>
date=<value>
delivery_feedback=<value>
product_feedback=<value>
packaging_feedback=<value>

After the six extracted fields, add these three assessment fields:

category=<delivery/product/packaging/service/other>
suggested_team=<logistics/product_support/customer_service>
proposed_action=<one brief recommendation for a human employee>

Choose the category based on the main issue.
Use logistics for delivery or packaging issues.
Use product_support for product faults.
Use customer_service if the issue is unclear.

These three fields are assessments, not extracted facts.
Do not promise refunds or replacements.


Rules:
- If information is missing, write "Not provided".
- Do not infer product damage from damaged packaging.
- If a complete date is given, use dd/mm/yyyy.
- If the year is missing, preserve the date as stated. Never invent a year.
- Do not add explanations, Markdown, or additional fields.
For the six extracted fields, write "Not provided" if the information is missing.
    
Customer message:
"I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

print(call_openai(extraction_prompt_v4))

clientid=Not provided  
itemnumber=12345  
date=15/03  
delivery_feedback=fast  
product_feedback=Not provided  
packaging_feedback=damaged  

category=logistics  
suggested_team=logistics  
proposed_action=Investigate packaging issue.


In [76]:
extraction_results_v4_15 = []

for i in range(15):
    result = call_openai(extraction_prompt_v4)
    extraction_results_v4_15.append(result)
    print(f"\n--- Test {i + 1} ---")
    print(result)


--- Test 1 ---
clientid=Not provided  
itemnumber=12345  
date=15/03  
delivery_feedback=Fast  
product_feedback=Not provided  
packaging_feedback=Damaged  

category=packaging  
suggested_team=logistics  
proposed_action=Investigate packaging issues.  

--- Test 2 ---
clientid=Not provided  
itemnumber=12345  
date=15/03  
delivery_feedback=Fast  
product_feedback=Not provided  
packaging_feedback=Damaged  

category=packaging  
suggested_team=logistics  
proposed_action=Investigate packaging issues.  

--- Test 3 ---
clientid=Not provided  
itemnumber=12345  
date=15/03/Not provided  
delivery_feedback=fast  
product_feedback=Not provided  
packaging_feedback=damaged  

category=logistics  
suggested_team=logistics  
proposed_action=Investigate packaging issues.

--- Test 4 ---
clientid=Not provided  
itemnumber=12345  
date=15/03  
delivery_feedback=Fast  
product_feedback=Not provided  
packaging_feedback=Damaged  

category=packaging  
suggested_team=logistics  
proposed_action=I

In [77]:
for i, result in enumerate(extraction_results_v4_15, start=1):
    date_line = next(
        (line.strip() for line in result.splitlines()
         if line.strip().startswith("date=")),
        "DATE FIELD MISSING"
    )
    print(f"Test {i}: {date_line}")

Test 1: date=15/03
Test 2: date=15/03
Test 3: date=15/03/Not provided
Test 4: date=15/03
Test 5: date=15/03
Test 6: date=15/03
Test 7: date=15/03/Not provided
Test 8: date=15/03
Test 9: date=15/03/Not provided
Test 10: date=15/03
Test 11: date=15/03
Test 12: date=15/03
Test 13: date=15/03
Test 14: date=15/03 Not provided
Test 15: date=15/03


In [85]:
extraction_prompt_v5 = """
Extract only information explicitly stated in the customer message.

Return exactly nine fields, one per line: six extracted fields followed by three assessment fields.clientid=<value>
itemnumber=<value>
date=<value>
delivery_feedback=<value>
product_feedback=<value>
packaging_feedback=<value>

After the six extracted fields, add these three assessment fields:

category=<delivery/product/packaging/service/other>
suggested_team=<logistics/product_support/customer_service>
proposed_action=<one brief recommendation for a human employee>

Choose the category based on the main issue.
Use logistics for delivery or packaging issues.
Use product_support for product faults.
Use customer_service if the issue is unclear.

These three fields are assessments, not extracted facts.
Do not promise refunds or replacements.


Rules:
- If information is missing, write "Not provided".
- Do not infer product damage from damaged packaging.
- Copy the date exactly as written in the customer message. Do not reformat it.
- Example: "I ordered on April 8th." → date=April 8th
- Never add a year that is not stated.
- Do not add explanations, Markdown, or additional fields.
For the six extracted fields, write "Not provided" if the information is missing.
    
Customer message:
Hi, I received item 67890 on ?. Delivery was two days late, but the product works perfectly and the packaging was intact. My client ID is unknown
"""

print(call_openai(extraction_prompt_v5))

clientid=Not provided  
itemnumber=67890  
date=Not provided  
delivery_feedback=Delivery was two days late  
product_feedback=The product works perfectly  
packaging_feedback=The packaging was intact  
category=delivery  
suggested_team=logistics  
proposed_action=Investigate delivery delays  


In [ ]:
extraction_results_v5_15 = [
    call_openai(extraction_prompt_v5)
    for _ in range(15)
]

date_passes = sum(
    "date=Not provided" in [line.strip() for line in result.splitlines()]
    for result in extraction_results_v5_15
)

print(f"Correcte datum: {date_passes}/15")
print(f"Percentage: {date_passes / 15:.1%}")

Correcte datum: 0/15
Percentage: 0.0%


In [91]:
product_prompt_final = """
Schrijf een korte Nederlandse productbeschrijving.
Gebruik uitsluitend de aangeleverde productgegevens.
Verzin geen eigenschappen, voordelen, betaalopties of levertijden.
Behoud de prijs en valuta. Gebruik valutacodes zoals USD en EUR.
Schrijf bij ontbrekende prijs of levertijd: Niet opgegeven.
De beschrijving bestaat uit één zin van maximaal 15 woorden.
Geef precies vier regels terug, zonder Markdown of extra tekst.

Voorbeeld 1
Productgegevens:
Product: Wired keyboard
Price: EUR 19.99
Delivery: 2 working days

Uitvoer:
PRODUCTNAAM: Bedraad toetsenbord
PRIJS: EUR 19.99
LEVERTIJD: 2 werkdagen
BESCHRIJVING: Bedraad toetsenbord.

Voorbeeld 2
Productgegevens:
Product: USB cable
Price: USD 5.00

Uitvoer:
PRODUCTNAAM: USB-kabel
PRIJS: USD 5.00
LEVERTIJD: Niet opgegeven
BESCHRIJVING: USB-kabel.

Maak nu de uitvoer voor deze productgegevens:
Product: Wireless mouse
Price: USD 29.99
"""

print(call_openai(product_prompt_final))

PRODUCTNAAM: Draadloze muis  
PRIJS: USD 29.99  
LEVERTIJD: Niet opgegeven  
BESCHRIJVING: Draadloze muis.


In [92]:
product_results_final_15 = [
    call_openai(product_prompt_final)
    for _ in range(15)
]

for i, result in enumerate(product_results_final_15, start=1):
    print(f"Test {i}: {result!r}")

Test 1: 'PRODUCTNAAM: Draadloze muis  \nPRIJS: USD 29.99  \nLEVERTIJD: Niet opgegeven  \nBESCHRIJVING: Draadloze muis.'
Test 2: 'PRODUCTNAAM: Draadloze muis  \nPRIJS: USD 29.99  \nLEVERTIJD: Niet opgegeven  \nBESCHRIJVING: Draadloze muis.'
Test 3: 'PRODUCTNAAM: Draadloze muis  \nPRIJS: USD 29.99  \nLEVERTIJD: Niet opgegeven  \nBESCHRIJVING: Draadloze muis.'
Test 4: 'PRODUCTNAAM: Draadloze muis  \nPRIJS: USD 29.99  \nLEVERTIJD: Niet opgegeven  \nBESCHRIJVING: Draadloze muis.  '
Test 5: 'PRODUCTNAAM: Draadloze muis  \nPRIJS: USD 29.99  \nLEVERTIJD: Niet opgegeven  \nBESCHRIJVING: Draadloze muis.'
Test 6: 'PRODUCTNAAM: Draadloze muis  \nPRIJS: USD 29.99  \nLEVERTIJD: Niet opgegeven  \nBESCHRIJVING: Draadloze muis.'
Test 7: 'PRODUCTNAAM: Draadloze muis  \nPRIJS: USD 29.99  \nLEVERTIJD: Niet opgegeven  \nBESCHRIJVING: Draadloze muis.'
Test 8: 'PRODUCTNAAM: Draadloze muis  \nPRIJS: USD 29.99  \nLEVERTIJD: Niet opgegeven  \nBESCHRIJVING: Draadloze muis.'
Test 9: 'PRODUCTNAAM: Draadloze muis  

In [96]:
extraction_prompt_final = extraction_prompt_v5 + """

Before producing the answer:
1. Check each extraction field against the customer message.
2. Mark missing information as Not provided.
3. Check that the date is copied exactly and packaging damage
   is not mistaken for product damage.
4. Apply the category and team rules.
5. Check the nine output fields.
Missing-value output must be exactly: Not provided
Never use ?, null, unknown, an empty value or another placeholder.

Example:
Message: "Item 555 arrived late."
date=Not provided
clientid=Not provided

Return only the final nine fields, without the intermediate reasoning.
"""

extraction_results_final_15 = [
    call_openai(extraction_prompt_final)
    for _ in range(15)
]

print(extraction_results_final_15[0])
print("Aantal antwoorden:", len(extraction_results_final_15))

clientid=Not provided  
itemnumber=67890  
date=Not provided  
delivery_feedback=Delivery was two days late  
product_feedback=The product works perfectly  
packaging_feedback=The packaging was intact  
category=delivery  
suggested_team=logistics  
proposed_action=Notify the logistics team about the delivery delay  
Aantal antwoorden: 15


In [98]:
expected = {
    "clientid": "Not provided",
    "itemnumber": "67890",
    "date": "Not provided",
    "category": "delivery",
    "suggested_team": "logistics",
}

passed_count = 0

for i, result in enumerate(extraction_results_final_15, start=1):
    fields = dict(
        line.strip().split("=", 1)
        for line in result.splitlines()
        if "=" in line
    )

    errors = [
        f"{key}: verwacht {value!r}, gekregen {fields.get(key)!r}"
        for key, value in expected.items()
        if fields.get(key) != value
    ]

    if len(fields) != 9:
        errors.append(f"{len(fields)} velden in plaats van 9")

    if errors:
        print(f"Test {i}: " + "; ".join(errors))
    else:
        passed_count += 1

print(f"\nAutomatische controles geslaagd: {passed_count}/15")
print(f"Percentage: {passed_count / 15:.1%}")


Automatische controles geslaagd: 15/15
Percentage: 100.0%


In [99]:
sentiment_positive = sentiment_prompt_v3.replace(
    "The delivery was fast, but the product stopped working after one day.",
    "The speaker sounds great and arrived on time. Very happy!"
)

sentiment_neutral = sentiment_prompt_v3.replace(
    "The delivery was fast, but the product stopped working after one day.",
    "My order number is 98765."
)

product_variation = product_prompt_final.replace(
    "Product: Wireless mouse\nPrice: USD 29.99",
    "Product: Wired headphones\nPrice: EUR 24.50\nDelivery: 4 working days"
)

print("POSITIEF:\n", call_openai(sentiment_positive))
print("\nNEUTRAAL:\n", call_openai(sentiment_neutral))
print("\nANDER PRODUCT:\n", call_openai(product_variation))

POSITIEF:
 label: positive  
description: The speaker sounds great and arrived on time.

NEUTRAAL:
 label: neutral  
description: The customer provides an order number without expressing an opinion.

ANDER PRODUCT:
 PRODUCTNAAM: Bedrade hoofdtelefoon  
PRIJS: EUR 24.50  
LEVERTIJD: 4 werkdagen  
BESCHRIJVING: Bedrade hoofdtelefoon.  


In [100]:
baseline_prompts = {
    "sentiment": sentiment_prompt_v1,
    "product": product_prompt_v1,
    "extraction": extraction_prompt_v1,
}

baseline_results = {}

for task, prompt in baseline_prompts.items():
    baseline_results[task] = {}

    for runs in (5, 10, 15):
        results = [call_openai(prompt) for _ in range(runs)]
        baseline_results[task][runs] = results
        print(f"{task}: {runs} antwoorden opgeslagen")

sentiment: 5 antwoorden opgeslagen
sentiment: 10 antwoorden opgeslagen
sentiment: 15 antwoorden opgeslagen
product: 5 antwoorden opgeslagen
product: 10 antwoorden opgeslagen
product: 15 antwoorden opgeslagen
extraction: 5 antwoorden opgeslagen
extraction: 10 antwoorden opgeslagen
extraction: 15 antwoorden opgeslagen
